# Heart Disease Prediction — Step 3: Preprocessing

This notebook gets the cleaned data into the exact shape a model needs:
1. Separate features (X) from the target (y)
2. Split into training and test sets
3. Scale the numeric features
4. Save everything so the next notebook (model training) can load it directly

No modelling happens here yet — this is purely preparation.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv('heart_disease_cleaned.csv')
print("Shape:", df.shape)
df.head()

Shape: (894, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalachh,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145.0,233.0,1.0,0.0,150.0,0.0,2.3,0.0,0.0,1.0,1
1,37,1,2,130.0,250.0,0.0,1.0,187.0,0.0,3.5,0.0,0.0,2.0,1
2,41,0,1,130.0,204.0,0.0,0.0,172.0,0.0,1.4,2.0,0.0,2.0,1
3,56,1,1,120.0,236.0,0.0,1.0,178.0,0.0,0.8,2.0,0.0,2.0,1
4,57,0,0,120.0,354.0,0.0,1.0,163.0,1.0,0.6,2.0,0.0,2.0,1


## 1. Do we need to encode the categorical columns?

Normally, "encoding" means converting text categories (like `"male"`/`"female"`) into numbers a model
can use. In our dataset, that's **already done** — every categorical column (`sex`, `cp`, `fbs`,
`restecg`, `exang`, `slope`, `ca`, `thal`) is already stored as a number code (e.g. `cp` is 0-3 for
the 4 chest pain types).

So there's **no extra encoding step needed** here — this was one thing your original pipeline plan
listed that turned out not to be necessary once we looked at the actual data.

## 2. Separate features (X) from the target (y)

`X` is everything the model is allowed to look at. `y` is what it's trying to predict.

In [2]:
X = df.drop(columns=['target'])
y = df['target']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (894, 13)
y shape: (894,)


## 3. Split into training and test sets

We hold back a portion of the data (**20%**) that the model never sees during training, so we have an
honest way to check how well it performs on patients it hasn't encountered before.

We use `stratify=y` so that both the training set and the test set keep the same ~55/45 disease
balance we saw in the EDA — otherwise, by chance, one set could end up with a very different mix than
the other.

`random_state=42` just makes the split reproducible — running this notebook again will always produce
the exact same split.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print()
print("Training set target balance:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Test set target balance:")
print(y_test.value_counts(normalize=True).round(3))

Training set: (715, 13)
Test set: (179, 13)

Training set target balance:
target
0    0.555
1    0.445
Name: proportion, dtype: float64

Test set target balance:
target
0    0.559
1    0.441
Name: proportion, dtype: float64


## 4. Scale the continuous features

Some models (like Logistic Regression, KNN, and SVM) are sensitive to features being on very different
scales — `chol` (values in the hundreds) would dominate over `oldpeak` (values under 7) just because
of its size, not because it's actually more important. Scaling puts every continuous feature on the
same footing.

Tree-based models like Random Forest and XGBoost don't actually need this — but since we're planning
to compare several models, it's simplest to prepare one scaled version we can use for all of them.

**Important:** we `fit` the scaler only on the training data, then apply it to both sets. If we fit it
on the full dataset (including the test set), information from the test set would leak into training —
which would make our evaluation results overly optimistic and not trustworthy.

In [4]:
continuous_cols = ['age', 'trestbps', 'chol', 'thalachh', 'oldpeak']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

X_train_scaled[continuous_cols].describe().round(2)

,age,trestbps,chol,thalachh,oldpeak
count,715.00,715.00,715.00,715.00,715.00
mean,-0.00,0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00
min,-2.60,-2.28,-2.90,-3.13,-0.83
25%,-0.78,-0.70,-0.66,-0.67,-0.83
50%,0.07,-0.14,-0.13,0.16,-0.29
75%,0.71,0.43,0.51,0.72,0.53
max,2.64,3.82,6.23,2.32,4.79


**Sanity check:** after scaling, each continuous column in the *training* set should have a mean
close to 0 and a standard deviation close to 1. The test set won't be exactly 0/1 — and that's
expected, since it was scaled using the training set's numbers, not its own.

## 5. Save everything for the next step

We save both the **unscaled** and **scaled** versions of the train/test sets, plus the fitted scaler
itself (so we could apply the exact same scaling to new patient data later, e.g. in the web app).

In [5]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
X_train_scaled.to_csv('X_train_scaled.csv', index=False)
X_test_scaled.to_csv('X_test_scaled.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

joblib.dump(scaler, 'scaler.joblib')

print("Saved: X_train.csv, X_test.csv, X_train_scaled.csv, X_test_scaled.csv, y_train.csv, y_test.csv, scaler.joblib")

Saved: X_train.csv, X_test.csv, X_train_scaled.csv, X_test_scaled.csv, y_train.csv, y_test.csv, scaler.joblib


## Summary

- No categorical encoding was needed — the data already arrived numerically coded
- Split into 80% training (715 patients) / 20% test (179 patients), keeping the same class balance
  in both
- Continuous features scaled using the training set only, to avoid leaking test-set information
- Both scaled and unscaled versions saved, so we can use whichever a given model needs

Next step: **model training** — we'll train a few different models (e.g. Random Forest, XGBoost) and
compare them.